# FPN-Mamba Experiments — A100 80GB Colab Runner

**Before starting:** Runtime → Change runtime type → A100 GPU.

**Run order:** Cell 1 → 2 → 3 → 4 → 5 (wandb) → 5.5 (delete stale results) → 6 → 7 → 8 → 9 → 10 → 11 → 12 (Fig. 2) → 13 → 14 (commit to git)

## Cell 1 — Verify A100 GPU

In [2]:
import torch

assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> GPU -> A100'
gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
bf16_ok  = torch.cuda.is_bf16_supported()

print(f'GPU  : {gpu_name}')
print(f'VRAM : {vram_gb:.1f} GB')
print(f'BF16 : {bf16_ok}  (True = A100 confirmed -- bfloat16 auto-enabled in trainer)')

if not bf16_ok:
    print('WARNING: Not an A100. Reduce batch sizes below if you get OOM.')

GPU  : NVIDIA A100-SXM4-80GB
VRAM : 85.1 GB
BF16 : True  (True = A100 confirmed -- bfloat16 auto-enabled in trainer)


## Cell 2 — Mount Google Drive

Checkpoints and results save here. Code and data come from GitHub, not Drive.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_RESULTS = '/content/drive/MyDrive/fpn_mamba/experiments'
os.makedirs(DRIVE_RESULTS, exist_ok=True)
print('Drive mounted. Results ->', DRIVE_RESULTS)

Mounted at /content/drive
Drive mounted. Results -> /content/drive/MyDrive/fpn_mamba/experiments


## Cell 3 — Clone Repo from GitHub

The dataset (4457 images) is committed in the repo — no separate download needed.

In [12]:
import os, sys

REPO_URL = 'https://github.com/Tech-sam-90/fpn-inceptentionnet'
REPO_DIR = '/content/fpn-inceptentionnet'
BRANCH   = 'version_2'

if not os.path.exists(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull origin {BRANCH}

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

DATA_ROOT = f'{REPO_DIR}/data'
classes   = sorted(d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d)))
n_images  = sum(len(os.listdir(os.path.join(DATA_ROOT, c))) for c in classes)

print(f'Repo    : {REPO_DIR}  (branch: {BRANCH})')
print(f'Data    : {DATA_ROOT}')
print(f'Classes : {classes}')
print(f'Images  : {n_images}')
!git log --oneline -3

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 2.14 KiB | 2.14 MiB/s, done.
From https://github.com/Tech-sam-90/fpn-inceptentionnet
   d6be67f..4496c57  version_2  -> origin/version_2
Already on 'version_2'
Your branch is behind 'origin/version_2' by 1 commit, and can be fast-forwarded.
  (use "git pull" to update your local branch)
From https://github.com/Tech-sam-90/fpn-inceptentionnet
 * branch            version_2  -> FETCH_HEAD
Updating d6be67f..4496c57
Fast-forward
 notebooks/colab_runner.ipynb | 1012 ++++++++++++++++++++----------------------
 1 file changed, 479 insertions(+), 533 deletions(-)
Repo    : /content/fpn-inceptentionnet  (branch: version_2)
Data    : /content/fpn-inceptentionnet/data
Classes : ['Astrocitoma', 'Carcinoma', 'Ependimoma', 'Ganglioglioma', 'Germinoma', 'Glioblastoma', 'Gran

## Cell 4 — Install Dependencies

In [5]:
!pip install -q timm einops scikit-learn scipy thop pyyaml tqdm matplotlib seaborn pandas wandb
print('Done.')

Done.


## Cell 5 — Weights & Biases Login

Paste your API key below. **Do not push this cell's key to GitHub.**
This cell runs only in Colab — the key never leaves your session.

In [6]:
import wandb

WANDB_API_KEY = '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'  # <-- replace with your key

wandb.login(key=WANDB_API_KEY, relogin=True)
print('wandb logged in. Project: medulloblastoma-classification')

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sadeniji to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb logged in. Project: medulloblastoma-classification


## Config Helper

Sets A100 80GB batch sizes: InceptentionNet stays at 8 (paper-exact), FPN-Mamba and ablation use 64.

In [7]:
import yaml
from pathlib import Path

def make_config(yaml_path: str, overrides: dict) -> str:
    def _deep_update(base, patch):
        for k, v in patch.items():
            if isinstance(v, dict) and k in base:
                _deep_update(base[k], v)
            else:
                base[k] = v
    with open(yaml_path) as f:
        cfg = yaml.safe_load(f)
    _deep_update(cfg, overrides)
    out = f'/tmp/{Path(yaml_path).stem}_patched.yaml'
    with open(out, 'w') as f:
        yaml.dump(cfg, f)
    return out

# Shared overrides for all experiments
A100_BASE = {
    'data':     {'data_root': DATA_ROOT},
    'training': {'num_workers': 4},  # A100 VMs have 12+ CPU cores
}

print('Config helper ready.')
print('  InceptentionNet batch : 8   (paper-exact, do not change)')
print('  FPN-Mamba batch       : 64  (A100 80GB comfortable at 64)')
print('  Ablation batch        : 64')

Config helper ready.
  InceptentionNet batch : 8   (paper-exact, do not change)
  FPN-Mamba batch       : 64  (A100 80GB comfortable at 64)
  Ablation batch        : 64


## Cell 5.5 — Delete Stale Pre-Fix Results (validation-leakage fix)

Wipes every run directory on Drive before retraining under the corrected
(calibration-split) protocol. This is not optional: `run_crossval()` resumes
**per fold** from `fold_{n}_result.json` before it ever rewrites `cv_results.json`,
so deleting only `cv_results.json` would silently reload the old, leaky
per-fold results instead of retraining. Run this once before Cells 6–9.

In [ ]:
import shutil, os

# Every run directory is wiped outright (not just cv_results.json / fold_*.pt)
# because run_crossval() resumes per FOLD from fold_{n}_result.json, before it
# ever re-aggregates cv_results.json. Leaving those per-fold files in place
# would silently reload pre-fix (leaky) results under the new protocol.
STALE_RUN_DIRS = {
    'InceptentionNet': f'{DRIVE_RESULTS}/inceptentionnet',
    'ResNet-50':       f'{DRIVE_RESULTS}/resnet50',
    'FPN-Mamba':       f'{DRIVE_RESULTS}/fpn_mamba',
    'Ablation':        f'{DRIVE_RESULTS}/ablation',
}

print('Wiping pre-fix (validation-leakage) results so nothing gets skipped via [RESUME]...')
for name, run_dir in STALE_RUN_DIRS.items():
    if os.path.exists(run_dir):
        shutil.rmtree(run_dir)
        print(f'  [DELETED] {name:<14} {run_dir}')
    else:
        print(f'  [SKIP]    {name:<14} {run_dir}  (nothing to delete)')

## Cell 6 — Train InceptentionNet Baseline

Paper-exact: LR=0.005, **batch=8**, 40 epochs, patience=10, sigma=2.0, 4x augmentation.
Batch size must stay at 8 to faithfully replicate the paper.

In [13]:
import os

BASELINE_RUN_DIR = f'{DRIVE_RESULTS}/inceptentionnet'
BASELINE_RESULTS = f'{BASELINE_RUN_DIR}/cv_results.json'

if os.path.exists(BASELINE_RESULTS):
    print('Already done. Delete', BASELINE_RESULTS, 'to retrain.')
else:
    cfg = make_config(
        f'{REPO_DIR}/configs/inceptentionnet.yaml',
        overrides={
            **A100_BASE,
            # batch_size stays at 8 -- paper-exact, do not override
            'output': {'run_dir': BASELINE_RUN_DIR},
        }
    )
    !python scripts/train.py --config {cfg}

print('\nBaseline results:', BASELINE_RESULTS)

Training: inceptentionnet
Run dir : /content/drive/MyDrive/fpn_mamba/experiments/inceptentionnet

Training : inceptentionnet
Device   : cuda (NVIDIA A100-SXM4-80GB)
Run dir  : /content/drive/MyDrive/fpn_mamba/experiments/inceptentionnet
wandb    : enabled

Dataset  : 736 samples  (106 MB / 630 non-MB)

  FOLD 1 / 5   (588 train / 148 val)
  [RESUME] Fold 1 already done — loading saved result.

  Fold 1 result -> AUC=0.6753  F1=0.6500  Sens=0.5909  Spec=0.9603  Acc=0.9054  Time=0.7min

  FOLD 2 / 5   (589 train / 147 val)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: sadeniji to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ setting up run fmh9lwgt (0.0s)
wandb: ⣻ setting up run fmh9lwgt (0.0s)
wandb: ⣽ setting up run fmh9lwgt (0.0s)
wandb: ⣾ setting up run fmh9lwgt (0.0s)
wandb: ⣷ setting up run fmh9lwgt (0.5s)
wandb: ⣯ setti

## Cell 7 — Train ResNet-50

A100 80GB: **batch=64**, timm `resnet50` backbone (ImageNet-pretrained),
freeze_layers=2. Same nested calibration-split protocol as every other model.

In [ ]:
import os

RESNET_RUN_DIR = f'{DRIVE_RESULTS}/resnet50'
RESNET_RESULTS = f'{RESNET_RUN_DIR}/cv_results.json'

if os.path.exists(RESNET_RESULTS):
    print('Already done. Delete', RESNET_RESULTS, 'to retrain.')
else:
    cfg = make_config(
        f'{REPO_DIR}/configs/resnet50.yaml',
        overrides={
            **A100_BASE,
            'training': {'batch_size': 64, 'num_workers': 4},
            'output':   {'run_dir': RESNET_RUN_DIR},
        }
    )
    !python scripts/train.py --config {cfg} --baseline_results {BASELINE_RESULTS}

print('\nResNet-50 results:', RESNET_RESULTS)

## Cell 8 — Train FPN-Mamba (Full Model)

A100 80GB: **batch=64**, bfloat16 auto-enabled, num_workers=4.

In [14]:
import os

FPN_RUN_DIR = f'{DRIVE_RESULTS}/fpn_mamba'
FPN_RESULTS = f'{FPN_RUN_DIR}/cv_results.json'

if os.path.exists(FPN_RESULTS):
    print('Already done. Delete', FPN_RESULTS, 'to retrain.')
else:
    cfg = make_config(
        f'{REPO_DIR}/configs/fpn_mamba.yaml',
        overrides={
            **A100_BASE,
            'training': {'batch_size': 64, 'num_workers': 4},
            'output':   {'run_dir': FPN_RUN_DIR},
        }
    )
    !python scripts/train.py --config {cfg} --baseline_results {BASELINE_RESULTS}

print('\nFPN-Mamba results:', FPN_RESULTS)

Already done. Delete /content/drive/MyDrive/fpn_mamba/experiments/fpn_mamba/cv_results.json to retrain.

FPN-Mamba results: /content/drive/MyDrive/fpn_mamba/experiments/fpn_mamba/cv_results.json


## Cell 9 — Ablation Study (5 variants, ~2 hrs total)

Each variant uses **batch=64** on A100 80GB.

In [9]:
import os

ABL_RUN_DIR = f'{DRIVE_RESULTS}/ablation'
ABL_RESULTS = f'{ABL_RUN_DIR}/ablation_summary.json'

cfg = make_config(
    f'{REPO_DIR}/configs/ablation.yaml',
    overrides={
        **A100_BASE,
        'training': {'batch_size': 64, 'num_workers': 4},
        'output':   {'run_dir': ABL_RUN_DIR},
    }
)
!python scripts/run_ablation.py --config {cfg}

print('\nAblation summary:', ABL_RESULTS)


  Variant: EfficientNet-B2 only
[RESUME] Results found at /content/drive/MyDrive/fpn_mamba/experiments/ablation/efficientnet_only/cv_results.json — replaying saved results.

Dataset  : 761 samples  (131 MB / 630 non-MB)

  FOLD 1 / 5   [loaded from Drive]

  Fold 1 result -> AUC=0.9553  F1=0.8462  Sens=0.8148  Spec=0.9762  Acc=0.9477  Time=3.2min

  FOLD 2 / 5   [loaded from Drive]

  Fold 2 result -> AUC=0.9753  F1=0.8387  Sens=1.0000  Spec=0.9206  Acc=0.9342  Time=3.0min

  FOLD 3 / 5   [loaded from Drive]

  Fold 3 result -> AUC=0.9048  F1=0.7119  Sens=0.8077  Spec=0.9048  Acc=0.8882  Time=1.8min

  FOLD 4 / 5   [loaded from Drive]

  Fold 4 result -> AUC=0.9444  F1=0.8000  Sens=0.8462  Spec=0.9444  Acc=0.9276  Time=3.4min

  FOLD 5 / 5   [loaded from Drive]

  Fold 5 result -> AUC=0.9606  F1=0.8000  Sens=0.9231  Spec=0.9206  Acc=0.9211  Time=2.6min

  Variant: + FPN (standard 3×3)
[RESUME] Results found at /content/drive/MyDrive/fpn_mamba/experiments/ablation/fpn_standard/cv_resul

## Cell 10 — Statistical Comparison (instant)

In [15]:
import json
from src.evaluation.stats import compare_models, print_comparison_table
from src.evaluation.metrics import summarize_folds

with open(BASELINE_RESULTS) as f: baseline = json.load(f)
with open(RESNET_RESULTS)   as f: resnet   = json.load(f)
with open(FPN_RESULTS)      as f: fpn      = json.load(f)

print('=== InceptentionNet (paper-exact re-run) ===')
for k, v in summarize_folds(baseline['fold_results']).items():
    print(f'  {k:<18}: {v["mean"]:.4f} +/- {v["std"]:.4f}')

print('\n=== ResNet-50 ===')
for k, v in summarize_folds(resnet['fold_results']).items():
    print(f'  {k:<18}: {v["mean"]:.4f} +/- {v["std"]:.4f}')

print('\n=== FPN-Mamba ===')
for k, v in summarize_folds(fpn['fold_results']).items():
    print(f'  {k:<18}: {v["mean"]:.4f} +/- {v["std"]:.4f}')

print('\n=== Statistical Comparison (bootstrap 95% CI + Wilcoxon p) ===')
table = compare_models(
    baseline['fold_results'], fpn['fold_results'],
    name_a='InceptentionNet', name_b='FPN-Mamba'
)
print_comparison_table(table, name_a='InceptentionNet', name_b='FPN-Mamba')

=== InceptentionNet (paper-exact re-run) ===
  accuracy          : 0.8600 +/- 0.0337
  precision         : 0.5359 +/- 0.1129
  recall            : 0.7658 +/- 0.1234
  sensitivity       : 0.7658 +/- 0.1234
  specificity       : 0.8762 +/- 0.0598
  f1                : 0.6146 +/- 0.0296
  auc               : 0.8297 +/- 0.0946
  fps               : 224.3229 +/- 4.6759
  training_time_sec : 108.1833 +/- 43.2717

=== FPN-Mamba ===
  accuracy          : 0.9803 +/- 0.0203
  precision         : 0.9235 +/- 0.0753
  recall            : 0.9695 +/- 0.0321
  sensitivity       : 0.9695 +/- 0.0321
  specificity       : 0.9825 +/- 0.0181
  f1                : 0.9454 +/- 0.0549
  auc               : 0.9935 +/- 0.0068
  fps               : 112.1760 +/- 0.8237
  training_time_sec : 262.1932 +/- 20.9956

=== Statistical Comparison (bootstrap 95% CI + Wilcoxon p) ===
Metric         InceptentionNet (95% CI)            FPN-Mamba (95% CI)              Delta   p-value  Sig?
-------------------------------------

## Cell 11 — Ablation Table (instant)

In [16]:
import json, pandas as pd

with open(ABL_RESULTS) as f:
    abl = json.load(f)

DISPLAY = {
    'efficientnet_only': 'EfficientNet-B2 only',
    'fpn_standard':      '+ FPN (standard 3x3)',
    'fpn_locality':      '+ LocalityMixing',
    'fpn_cross_mamba':   '+ Cross-scale Mamba',
    'fpn_mamba_full':    '+ GeM + SE  (full model)',
}
METRICS = ['accuracy', 'precision', 'recall', 'sensitivity', 'specificity', 'f1', 'auc']

rows = []
for variant, name in DISPLAY.items():
    if variant not in abl: continue
    row = {'Variant': name}
    for m in METRICS:
        mu  = abl[variant].get(m, {}).get('mean', float('nan'))
        std = abl[variant].get(m, {}).get('std',  float('nan'))
        row[m] = f'{mu:.4f} +/- {std:.4f}'
    rows.append(row)

df = pd.DataFrame(rows).set_index('Variant')
print(df.to_string())

                                   accuracy          precision             recall        sensitivity        specificity                 f1                auc
Variant                                                                                                                                                      
EfficientNet-B2 only      0.9238 +/- 0.0222  0.7406 +/- 0.0897  0.8783 +/- 0.0820  0.8783 +/- 0.0820  0.9333 +/- 0.0278  0.7993 +/- 0.0534  0.9481 +/- 0.0266
+ FPN (standard 3x3)      0.9803 +/- 0.0093  0.9427 +/- 0.0517  0.9464 +/- 0.0440  0.9464 +/- 0.0440  0.9873 +/- 0.0120  0.9431 +/- 0.0256  0.9912 +/- 0.0058
+ LocalityMixing          0.9750 +/- 0.0150  0.8819 +/- 0.0639  0.9923 +/- 0.0172  0.9923 +/- 0.0172  0.9714 +/- 0.0174  0.9329 +/- 0.0385  0.9944 +/- 0.0041
+ Cross-scale Mamba       0.9698 +/- 0.0178  0.8703 +/- 0.0784  0.9772 +/- 0.0208  0.9772 +/- 0.0208  0.9683 +/- 0.0217  0.9190 +/- 0.0449  0.9913 +/- 0.0056
+ GeM + SE  (full model)  0.9803 +/- 0.0203  0.9235 

## Cell 12 — Figure 2: Aggregate Confusion Matrix (FPN-Mamba)

Built from `cv_results['aggregate_confusion_matrix']` (summed TP/TN/FP/FN across
the 5 held-out folds, each already thresholded on its own calibration split) —
**not** by pooling probabilities and re-thresholding at 0.5. This guarantees the
plotted matrix and the text-reported sensitivity/specificity can never diverge.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

with open(FPN_RESULTS) as f:
    fpn = json.load(f)

agg = fpn['aggregate_confusion_matrix']
tp, tn, fp, fn = agg['tp'], agg['tn'], agg['fp'], agg['fn']

sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

print('=== FPN-Mamba Aggregate Confusion Matrix (5-fold held-out, calibration-selected thresholds) ===')
print(f'  TP={tp}  TN={tn}  FP={fp}  FN={fn}')
print(f'  Sensitivity = {sensitivity:.4f}')
print(f'  Specificity = {specificity:.4f}')
print('  (Compare against the mean sensitivity/specificity printed in Cell 10 --')
print('   both are derived from the same per-fold calibration-thresholded predictions.)')

cm = np.array([[tn, fp], [fn, tp]])
fig, ax = plt.subplots(figsize=(5, 4.5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1]); ax.set_xticklabels(['Non-MB', 'MB'])
ax.set_yticks([0, 1]); ax.set_yticklabels(['Non-MB', 'MB'])
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('FPN-Mamba — Aggregate Confusion Matrix (5-fold CV)')
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=14)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()

FIG2_PATH = f'{DRIVE_RESULTS}/figure2_confusion_matrix.png'
fig.savefig(FIG2_PATH, dpi=200)
print(f'\nSaved -> {FIG2_PATH}')
plt.show()

## Cell 13 — Verify Drive (all checkpoints saved)

In [ ]:
import os

def check(path, label):
    ok   = os.path.exists(path)
    size = f'{os.path.getsize(path)/1024:.0f} KB' if ok else ''
    print(f"  {'[OK]' if ok else '[MISSING]':<10} {label:<40} {size}")

print('=== Drive Contents ===')
check(BASELINE_RESULTS, 'InceptentionNet cv_results.json')
check(RESNET_RESULTS,   'ResNet-50 cv_results.json')
check(FPN_RESULTS,      'FPN-Mamba cv_results.json')
check(ABL_RESULTS,      'Ablation summary.json')
for run_name, run_dir in [('InceptentionNet', BASELINE_RUN_DIR), ('ResNet-50', RESNET_RUN_DIR), ('FPN-Mamba', FPN_RUN_DIR)]:
    for fold in range(1, 6):
        check(f'{run_dir}/fold_{fold}.pt', f'{run_name} fold_{fold}.pt')
    check(f'{run_dir}/best_model.pt', f'{run_name} best_model.pt')

## Cell 14 — Commit Results to Git

Copies each model's `cv_results.json` (and the ablation summary + per-variant
results) from Drive into `experiments/runs/<model>/` inside the cloned repo and
commits them. This is the step that was skipped for ResNet-50 last time, which is
why Table 1's std-dev cells for it went missing — the manuscript was typed up from
a Drive-only printout instead of a committed, reproducible `cv_results.json`.

Does **not** push automatically — review the commit, then run the printed
`git push` command yourself.

In [ ]:
import shutil, os, subprocess

COMMIT_TARGETS = {
    'inceptentionnet': (BASELINE_RUN_DIR, f'{REPO_DIR}/experiments/runs/inceptentionnet'),
    'resnet50':        (RESNET_RUN_DIR,   f'{REPO_DIR}/experiments/runs/resnet50'),
    'fpn_mamba':       (FPN_RUN_DIR,      f'{REPO_DIR}/experiments/runs/fpn_mamba'),
}

os.chdir(REPO_DIR)
added_paths = []
for name, (drive_dir, repo_dir) in COMMIT_TARGETS.items():
    src = f'{drive_dir}/cv_results.json'
    if not os.path.exists(src):
        print(f'  [MISSING] {name}: {src} not found -- run its training cell first.')
        continue
    os.makedirs(repo_dir, exist_ok=True)
    dst = f'{repo_dir}/cv_results.json'
    shutil.copyfile(src, dst)
    added_paths.append(dst)
    print(f'  [COPIED] {src} -> {dst}')

# Ablation: the summary table plus each variant's own cv_results.json
# (includes 'efficientnet_only' == EfficientNet-B2+GAP in the paper).
abl_root_drive = ABL_RUN_DIR
abl_root_repo  = f'{REPO_DIR}/experiments/runs/ablation'
if os.path.exists(f'{abl_root_drive}/ablation_summary.json'):
    os.makedirs(abl_root_repo, exist_ok=True)
    shutil.copyfile(f'{abl_root_drive}/ablation_summary.json', f'{abl_root_repo}/ablation_summary.json')
    added_paths.append(f'{abl_root_repo}/ablation_summary.json')
    for variant in ['efficientnet_only', 'fpn_standard', 'fpn_locality', 'fpn_cross_mamba', 'fpn_mamba_full']:
        src = f'{abl_root_drive}/{variant}/cv_results.json'
        if os.path.exists(src):
            dst_dir = f'{abl_root_repo}/{variant}'
            os.makedirs(dst_dir, exist_ok=True)
            dst = f'{dst_dir}/cv_results.json'
            shutil.copyfile(src, dst)
            added_paths.append(dst)
            print(f'  [COPIED] {src} -> {dst}')
else:
    print(f'  [MISSING] ablation: {abl_root_drive}/ablation_summary.json not found -- run the ablation cell first.')

if added_paths:
    rel_paths = [os.path.relpath(p, REPO_DIR) for p in added_paths]
    subprocess.run(['git', 'add'] + rel_paths, check=True)
    status = subprocess.run(['git', 'status', '--short'] + rel_paths, capture_output=True, text=True).stdout
    print('\ngit status (staged):\n', status)
    subprocess.run(['git', 'commit', '-m',
                     'Add post-leakage-fix cv_results.json for all four models (5-fold CV, calibration-split protocol)'],
                    check=True)
    print('\nCommitted locally. Review with `git log -1 -p`, then push yourself:')
    print(f'  cd {REPO_DIR} && git push origin {BRANCH}')
else:
    print('\nNothing to commit -- no cv_results.json files were found.')